In [ ]:
import json, torch
import matplotlib.pyplot as plt
from torch.utils.data import DataLoader
from src.data import load_dataset
from src.evaluate import evaluate_model, print_results

In [ ]:
METHOD       = "tesnet" # change per member
CHECKPOINT   = "checkpoints/tesnet/tesnet_k32_best.pt" # change per member
CONFIG_FILE  = "checkpoints/tesnet/tesnet_k32_config.json" # change per member
DEVICE       = "cuda" if torch.cuda.is_available() else "cpu"
NUM_IMAGES   = 8 # how many test images to visualize

In [ ]:
config = json.load(open(CONFIG_FILE))

if METHOD == "tesnet":
  from src.models.tesnet import TesNet
  model = TesNet(num_classes=200, num_concepts=config["num_concepts"])
elif METHOD == "baseline":
  from src.models import BaselineModel
  model = BaselineModel(num_classes=200)
# etc. — each member fills in their branch

ckpt = torch.load(CHECKPOINT, map_location=DEVICE)
model.load_state_dict(ckpt["model_state"])
model.eval().to(DEVICE)
print(f"Loaded epoch {ckpt['epoch']} checkpoint")

In [ ]:
test_loader = DataLoader(load_dataset("cub200", "test"), batch_size=64, num_workers=4)
results = evaluate_model(model, test_loader, DEVICE)
print_results(results, model_name=METHOD)
print(f"per-class std: {results['per_class_accuracy'].std():.4f}")

In [ ]:
activation_key = {
  "tesnet":    "concept_scores",
  "protopnet": "prototype_similarities",
  "prototree": "routing_probs", # member C should adjust this
  "pipnet":    "active_prototypes",
}[METHOD]

scores = []
with torch.no_grad():
  for images, _ in test_loader:
      out = model.explain(images.to(DEVICE))
      scores.append(out[activation_key].cpu())

mean_activation = torch.cat(scores).mean().item()
print(f"mean activation score: {mean_activation:.4f}")

In [ ]:
# TODO: visualization is method-specific, needs to be adapted accordingly per member
# show concept activation maps for a batch of test images
images, labels = next(iter(DataLoader(load_dataset("cub200", "test"), batch_size=NUM_IMAGES)))
with torch.no_grad():
    out = model.explain(images.to(DEVICE))

concept_maps = out["concept_maps"].cpu()   # (B, K, 7, 7)
scores       = out["concept_scores"].cpu() # (B, K)

fig, axes = plt.subplots(NUM_IMAGES, 2, figsize=(8, NUM_IMAGES * 3))
for i, img in enumerate(images):
  # original image
  axes[i, 0].imshow(img.permute(1, 2, 0).numpy() * 0.5 + 0.5)
  axes[i, 0].axis("off")
  # most active concept's spatial map
  top_concept = scores[i].argmax().item()
  heatmap = concept_maps[i, top_concept]
  axes[i, 1].imshow(torch.nn.functional.interpolate(
      heatmap[None, None], size=(224, 224), mode="bilinear"
  ).squeeze().numpy(), cmap="hot")
  axes[i, 1].set_title(f"concept {top_concept}  score={scores[i, top_concept]:.2f}")
  axes[i, 1].axis("off")
plt.tight_layout()